# Deepfake Detection: CNN + ViT + FFT + Temporal LSTM

This notebook upgrades the frame-only hybrid model to a **video-level** model using:
- Spatial features: ResNet50 + ViT
- Frequency features: FFT artifact branch
- Temporal modeling: BiLSTM over frame sequences

It also includes tunable fraction controls so you can train on smaller subsets first.

In [1]:
import os
import re
import time
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
import matplotlib.pyplot as plt

In [ ]:
# =========================
# Config (tune these)
# =========================
SEED = 42
DATA_PATH = Path("../data/FF_frames")

# Fraction controls
FRACTION = 0.10          # train fraction (0 < FRACTION <= 1)
VAL_FRACTION = 1.00      # validation fraction (use 1.0 for full val)

VAL_RATIO = 0.20         # video-level split
SEQ_LEN = 8              # frames per sequence
SEQ_STRIDE = 4           # window stride when generating sequences

BATCH_SIZE = 4
NUM_WORKERS = 0          # Windows-safe
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
FREEZE_BACKBONES = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Using device:", DEVICE)
print("Data path:", DATA_PATH)
print("FRACTION:", FRACTION, "VAL_FRACTION:", VAL_FRACTION)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# =========================
# Data indexing helpers
# =========================
_FRAME_RE = re.compile(r"^(.*)_(\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)

def parse_video_id_and_frame_idx(filename: str):
    m = _FRAME_RE.match(filename)
    if not m:
        return None, None
    video_id = m.group(1)
    frame_idx = int(m.group(2))
    return video_id, frame_idx

def build_video_records(data_path: Path, min_frames: int = 8):
    class_names = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    # key -> record
    # key format: class_name/video_id
    video_map = {}

    for class_name in class_names:
        class_dir = data_path / class_name
        for p in class_dir.glob("*"):
            if not p.is_file():
                continue
            video_id, frame_idx = parse_video_id_and_frame_idx(p.name)
            if video_id is None:
                continue

            key = f"{class_name}/{video_id}"
            if key not in video_map:
                video_map[key] = {
                    "video_id": video_id,
                    "label_name": class_name,
                    "label": class_to_idx[class_name],
                    "frames": [],
                }

            video_map[key]["frames"].append((frame_idx, p))

    video_records = []
    for rec in video_map.values():
        rec["frames"] = [p for _, p in sorted(rec["frames"], key=lambda x: x[0])]
        if len(rec["frames"]) >= min_frames:
            video_records.append(rec)

    return video_records, class_names, class_to_idx

def split_video_records(video_records, val_ratio=0.2, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for rec in video_records:
        by_label[rec["label"]].append(rec)

    train_records, val_records = [], []
    for _, recs in by_label.items():
        rng.shuffle(recs)
        n_val = max(1, int(len(recs) * val_ratio))
        val_records.extend(recs[:n_val])
        train_records.extend(recs[n_val:])

    rng.shuffle(train_records)
    rng.shuffle(val_records)
    return train_records, val_records

def make_sequence_samples(video_records, seq_len=8, seq_stride=4):
    samples = []
    for rec in video_records:
        frames = rec["frames"]
        for start in range(0, len(frames) - seq_len + 1, seq_stride):
            seq_paths = frames[start:start + seq_len]
            samples.append({
                "video_id": rec["video_id"],
                "label": rec["label"],
                "label_name": rec["label_name"],
                "frame_paths": seq_paths,
            })
    return samples

def apply_fraction_balanced(samples, fraction=1.0, seed=42):
    if fraction >= 1.0:
        return samples
    if fraction <= 0:
        raise ValueError("fraction must be > 0")

    rng = random.Random(seed)
    by_label = defaultdict(list)
    for s in samples:
        by_label[s["label"]].append(s)

    picked = []
    for _, group in by_label.items():
        rng.shuffle(group)
        n = max(1, int(len(group) * fraction))
        picked.extend(group[:n])

    rng.shuffle(picked)
    return picked

def count_labels(samples, class_names):
    counts = {name: 0 for name in class_names}
    for s in samples:
        counts[class_names[s["label"]]] += 1
    return counts

class FrameSequenceDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rec = self.samples[idx]
        frames = []
        for path in rec["frame_paths"]:
            img = Image.open(path).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
            frames.append(img)

        x = torch.stack(frames, dim=0)  # [T, C, H, W]
        y = torch.tensor(rec["label"], dtype=torch.long)
        return x, y

In [ ]:
video_records, class_names, class_to_idx = build_video_records(DATA_PATH, min_frames=SEQ_LEN)
train_videos, val_videos = split_video_records(video_records, val_ratio=VAL_RATIO, seed=SEED)

train_samples = make_sequence_samples(train_videos, seq_len=SEQ_LEN, seq_stride=SEQ_STRIDE)
val_samples = make_sequence_samples(val_videos, seq_len=SEQ_LEN, seq_stride=SEQ_STRIDE)

train_samples = apply_fraction_balanced(train_samples, fraction=FRACTION, seed=SEED)
val_samples = apply_fraction_balanced(val_samples, fraction=VAL_FRACTION, seed=SEED)

train_dataset = FrameSequenceDataset(train_samples, transform=train_transform)
val_dataset = FrameSequenceDataset(val_samples, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Classes:", class_names)
print("Videos total:", len(video_records))
print("Train videos:", len(train_videos), "Val videos:", len(val_videos))
print("Train sequences:", len(train_dataset), "Val sequences:", len(val_dataset))
print("Train class balance:", count_labels(train_samples, class_names))
print("Val class balance:", count_labels(val_samples, class_names))

In [ ]:
# =========================
# Model: Spatial + FFT + Temporal
# =========================
class FFTBranch(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # x: [B, T, C, H, W]
        b, t, c, h, w = x.shape

        gray = x.mean(dim=2)  # [B, T, H, W]
        fft = torch.fft.fft2(gray, norm="ortho")
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        mag = torch.log1p(torch.abs(fft))

        mag = mag.unsqueeze(2)  # [B, T, 1, H, W]
        mag = mag.reshape(b * t, 1, h, w)

        feat = self.net(mag)
        feat = feat.reshape(b, t, -1)
        return feat

class TemporalHybridModel(nn.Module):
    def __init__(
        self,
        num_classes=2,
        fft_dim=128,
        lstm_hidden=256,
        freeze_backbones=True,
    ):
        super().__init__()
        self.freeze_backbones = freeze_backbones

        # Spatial backbones
        self.cnn = timm.create_model("resnet50", pretrained=True, num_classes=0, global_pool="avg")
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)

        if self.freeze_backbones:
            for p in self.cnn.parameters():
                p.requires_grad = False
            for p in self.vit.parameters():
                p.requires_grad = False

        self.fft_branch = FFTBranch(out_dim=fft_dim)

        # 2048 (resnet50) + 768 (vit_base) + fft_dim
        temporal_in_dim = 2048 + 768 + fft_dim

        self.temporal = nn.LSTM(
            input_size=temporal_in_dim,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        # x: [B, T, C, H, W]
        b, t, c, h, w = x.shape
        x_bt = x.reshape(b * t, c, h, w)

        if self.freeze_backbones:
            with torch.no_grad():
                cnn_feat = self.cnn(x_bt)
                vit_feat = self.vit(x_bt)
        else:
            cnn_feat = self.cnn(x_bt)
            vit_feat = self.vit(x_bt)

        cnn_feat = cnn_feat.reshape(b, t, -1)
        vit_feat = vit_feat.reshape(b, t, -1)
        fft_feat = self.fft_branch(x)

        seq_feat = torch.cat([cnn_feat, vit_feat, fft_feat], dim=-1)
        lstm_out, _ = self.temporal(seq_feat)

        # Use last time-step representation
        temporal_repr = lstm_out[:, -1, :]
        logits = self.classifier(temporal_repr)
        return logits

def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
model = TemporalHybridModel(
    num_classes=len(class_names),
    fft_dim=128,
    lstm_hidden=256,
    freeze_backbones=FREEZE_BACKBONES,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params, trainable_params = count_trainable_params(model)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    loop = tqdm(loader, desc="Training", leave=False)
    for x, y in loop:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_pos_probs = []

    loop = tqdm(loader, desc="Validation", leave=False)
    for x, y in loop:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

        if probs.shape[1] == 2:
            all_pos_probs.extend(probs[:, 1].detach().cpu().numpy().tolist())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, all_labels, all_preds, all_pos_probs

In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

best_val_acc = 0.0
best_ckpt = "small_cnn_vit_fft_lstm_model.pth"

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"Epoch Time: {(time.time() - start) / 60:.2f} min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "config": {
                "seq_len": SEQ_LEN,
                "seq_stride": SEQ_STRIDE,
                "fraction": FRACTION,
                "val_fraction": VAL_FRACTION,
                "freeze_backbones": FREEZE_BACKBONES,
            },
        }, best_ckpt)
        print("Saved best model ->", best_ckpt)

print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

In [ ]:
# Final validation metrics
val_loss, val_acc, labels_all, preds_all, probs_all = evaluate(model, val_loader, criterion, DEVICE)

precision = precision_score(labels_all, preds_all, average="weighted", zero_division=0)
recall = recall_score(labels_all, preds_all, average="weighted", zero_division=0)
f1 = f1_score(labels_all, preds_all, average="weighted", zero_division=0)

print("\n===== MODEL PERFORMANCE =====")
print("Accuracy:", val_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(labels_all, preds_all, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(labels_all, preds_all)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# ROC (binary only)
if len(class_names) == 2 and len(probs_all) == len(labels_all):
    fpr, tpr, _ = roc_curve(labels_all, probs_all)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()
else:
    print("ROC is only shown for binary classification.")